## Gold Product Validation

Valida os produtos analíticos da camada Gold contra seus contratos e casos de uso, combinando verificações objetivas com evidências analíticas.

Cada produto é analisado a partir do significado de seus dados, verificando cardinalidade, unicidade, regras de negócio e métricas relevantes antes de demonstrar como pode responder às perguntas analíticas para as quais foi projetado.

As visualizações são utilizadas quando ajudam a interpretar a evidência; não substituem as validações objetivas.

## Step 1. Imports

In [0]:
from pyspark.sql import functions as F

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import GOLD_CONTRACTS

## Step 2. Configuração

Define o namespace Gold que contém os produtos analíticos materializados e que serão avaliados neste notebook.

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("gold_schema", "")

In [0]:
config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    gold_schema=dbutils.widgets.get("gold_schema"),
)

## Step 3. Leitura Gold

Carrega os produtos materializados pela camada Gold.

A validação parte das tabelas efetivamente entregues pela aplicação, e não da reconstrução das transformações que as produziram. Assim, as evidências deste notebook representam o estado real disponível para consumo analítico.

In [0]:
gold_dataframes = {
    contract.name: spark.table(
        f"{config.gold_namespace}.{contract.name}"
    )
    for contract in GOLD_CONTRACTS
}

In [0]:
movie_performance_df = gold_dataframes["movie_performance"]
movie_genre_performance_df = gold_dataframes["movie_genre_performance"]
movie_credit_participation_df = gold_dataframes["movie_credit_participation"]
movie_company_performance_df = gold_dataframes["movie_company_performance"]
movie_country_performance_df = gold_dataframes["movie_country_performance"]
movie_language_profile_df = gold_dataframes["movie_language_profile"]
movie_keyword_performance_df = gold_dataframes["movie_keyword_performance"]

## Step 4. Validação dos produtos Gold

Avalia cada produto Gold contra seu contrato e sua finalidade analítica.

As verificações combinam evidências objetivas de qualidade e regras de negócio com análises que demonstram como os dados podem ser interpretados pelos seus consumidores

### 4.1 movie_performance

Representa cada filme individualmente e reúne seus principais atributos, indicadores de audiência e métricas comerciais.
Antes de utilizar essas métricas em análises, verificamos se cada filme aparece uma única vez e se o produto mantém a estrutura definida em seu contrato

In [0]:
# busca e retorna o primeiro contraor da lista correspondente a 'movie_performance'
movie_performance_contract = next(
    (contract for contract in GOLD_CONTRACTS if contract.name == "movie_performance"),
    None,
)

row_count = movie_performance_df.count()

duplicate_key_count = (
    movie_performance_df.groupBy(*movie_performance_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_performance_df.schema.fields
]

expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_performance_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError("movie_performance possui mais de uma linha para o mesmo filme.")

if not schema_matches:
    raise RuntimeError(
        "movie_performance não corresponde ao schema definido no contrato Gold."
    )

#### 4.1.2 Regras e métricas

Valida se as métricas derivadas de cada filme seguem as regras comerciais definidas para o produto.

Um filme é elegível para métricas comerciais quando possui orçamento maior que zero. Para esses filmes, lucro e ROI devem refletir orçamento e receita.
Filmes sem orçamento válido permanecem no produto, mas não recebem essas métricas.

O ano de lançamento também é confrontado com a data de lançamento registrada.

In [0]:
# definição das condições de validação (regras booleanas)
has_budget = F.col("budget") > 0

# regra: elegibilidade comercial deve ser idêntica a ter budget > 0
cond_eligibility = F.col("commercial_metrics_eligible") != has_budget

# regra: filmes inelegíveis (budget <= 0) não podem ter profit nem roi preenchidos
cond_ineligible = (~has_budget) & (
    F.col("profit").isNotNull() | F.col("roi").isNotNull()
)

# regra: ano de lançamento deve bater com o ano da data
cond_release_year = F.col("release_year") != F.year("release_date")

# regra: lucro deve ser igual a receita menos orçamento
cond_profit = has_budget & (F.col("profit") != (F.col("revenue") - F.col("budget")))

# regra: ROI = (receita - orçamento) / orçamento
# 1e-9 (0.000000001): tolerância de ponto flutuante para ignorar resíduos de arredondamento
expected_roi = (F.col("revenue") - F.col("budget")) / F.col("budget")
cond_roi = has_budget & (F.abs(F.col("roi") - expected_roi) > 1e-9)

# agregação e contagem de violações
movie_metric_violations_df = movie_performance_df.select(
    F.sum(cond_eligibility.cast("int")).alias("eligibility_violations"),
    F.sum(cond_ineligible.cast("int")).alias("ineligible_metrics_violations"),
    F.sum(cond_release_year.cast("int")).alias("release_year_violations"),
    F.sum(cond_profit.cast("int")).alias("profit_violations"),
    F.sum(cond_roi.cast("int")).alias("roi_violations"),
)

display(movie_metric_violations_df)

#### 4.1.3 Evidência analítica

Com a integridade e as regras comerciais validadas, o produto pode ser utilizado para comparar o desempenho individual dos filmes.

A primeira análise considera apenas filmes elegíveis para métricas comerciais, pois lucro e ROI dependem da existência de um orçamento válido.

A visão a seguir destaca os filmes com maior receita e permite observar, simultaneamente, orçamento e lucro, preservando o contexto necessário para
interpretar o resultado comercial.

In [0]:
top_movie_performance_df = (
    movie_performance_df.filter(F.col("commercial_metrics_eligible"))
    .select(
        "title",
        "release_year",
        "budget",
        "revenue",
        "profit",
        "roi",
        "popularity",
        "vote_average",
        "vote_count",
    )
    .orderBy(F.col("revenue").desc())
    .limit(10)
)

display(top_movie_performance_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** Os filmes de maior receita também operam com volumes expressivos de orçamento, mas a relação entre investimento e retorno não é uniforme. Produções como *Minions* e *Jurassic World*, por exemplo, atingem receitas comparáveis às de outros títulos do recorte com frações de seus orçamentos.

> O ranking por receita, portanto, não deve ser interpretado como um ranking de eficiência comercial. O produto preserva orçamento, receita, lucro e ROI no mesmo grão de filme, permitindo analisar tanto escala financeira quanto retorno relativo sem reconstruir essas métricas a partir das fontes Silver.


In [0]:
top_movie_profit_df = (
    movie_performance_df.filter(F.col("commercial_metrics_eligible"))
    .select(
        "title",
        "release_year",
        "budget",
        "revenue",
        "profit",
        "roi",
    )
    .orderBy(F.col("profit").desc())
    .limit(10)
)

display(top_movie_profit_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** O ranking por lucro mantém *Avatar* e *Titanic* nas primeiras posições, mas altera a ordenação observada no ranking de receita.
> *Jurassic World* e *Furious 7*, por exemplo, avançam em relação a *The Avengers* quando o resultado passa a considerar a diferença entre receita e orçamento.
>
> A comparação demonstra que liderança em receita não determina, por si só, a mesma posição em lucro, reforçando a utilidade de preservar ambas as métricas no produto Gold

In [0]:
roi_relevance_budget = 10_000_000

top_movie_roi_df = (
    movie_performance_df.filter(
        F.col("commercial_metrics_eligible") & (F.col("budget") > roi_relevance_budget)
    )
    .withColumn("roi_percent", F.col("roi") * 100)
    .select(
        "title",
        "release_year",
        "budget",
        "revenue",
        "profit",
        "roi",
        "roi_percent"
    )
    .orderBy(F.col("roi_percent").desc())
    .limit(10)
)

display(top_movie_roi_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** A análise de eficiência comercial por ROI revela um ranking distinto dos líderes em receita e lucro. *E.T. the Extra-Terrestrial* e *Star Wars* aparecem com retornos superiores a 6900%, enquanto outros títulos do recorte também apresentam retornos muito superiores ao valor originalmente investido.
>
> Para esta análise foi aplicado um critério de relevância de `budget > 10M`. Esse filtro pertence exclusivamente ao recorte analítico e não ao contrato Gold, sua finalidade é reduzir a influência de produções de orçamento muito baixo, cujo pequeno denominador pode produzir valores extremos de ROI.

### 4.2 movie_genre_performance

Relaciona cada filme aos seus gêneros e preserva, em cada associação, as métricas necessárias para analisar desempenho comercial e audiência por gênero.

Como um filme pode pertencer a vários gêneros, o produto possui grão de associação `movie_id × genre_id`. Antes das análises agregadas, verificamos se cada associação ocorre uma única vez e se a estrutura materializada permanece compatível com o contrato Gold.

#### 4.2.1 Integridade do produto

Verifica se o produto preserva uma única linha por associação entre filme e gênero e se sua estrutura corresponde ao contrato Gold definido para `movie_genre_performance`. Essa validação é especialmente importante porque análises posteriores agregarão métricas no nível de gênero sobre um relacionamento naturalmente muitos-para-muitos.

In [0]:
# busca e retorna o primeiro contrato da lista correspondente a 'movie_genre_performance'
movie_genre_contract = next(
    contract
    for contract in GOLD_CONTRACTS
    if contract.name == "movie_genre_performance"
)

# cardinalidade observado no produto
row_count = movie_genre_performance_df.count()

# regra: cada associação movie_id x genre_id deve ocorrer uma única vez
duplicate_key_count = (
    movie_genre_performance_df.groupBy(*movie_genre_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# estrutura materialziada obsevada
actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_genre_performance_df.schema.fields
]

# estrutura definida pelo contrato da Gold
expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_genre_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError(
        "movie_genre_performance possui associações filme × gênero duplicadas."
    )

if not schema_matches:
    raise RuntimeError(
        "movie_genre_performance não corresponde ao schema definido no contrato Gold."
    )

#### 4.2.2 Regras e métricas

Valida se as métricas comerciais replicadas em cada associação entre filme e gênero preservam as regras definidas para os produtos Gold.

Como um mesmo filme pode pertencer a múltiplos gêneros, suas métricas aparecem em mais de uma associação. Essa característica é intencional e permite
análises por gênero, mas exige que agregações posteriores respeitem o grão muitos-para-muitos do produto.

In [0]:
# definição das condições de validação (regras booleanas)
has_budget = F.col("budget") > 0

# regra: elegibilidade comercial deve ser idêntica a ter budget > 0
cond_eligibility = F.col("commercial_metrics_eligible") != has_budget

# regra: associações de filmes inelegíveis não podem ter profit nem roi preenchidos
cond_ineligible = (~has_budget) & (
    F.col("profit").isNotNull() | F.col("roi").isNotNull()
)

# regra: ano de lançamento deve bater com o ano da data
cond_release_year = F.col("release_year") != F.year("release_date")

# regra: lucro deve ser igual a receita menos orçamento
cond_profit = has_budget & (F.col("profit") != (F.col("revenue") - F.col("budget")))

# regra: ROI = (receita - orçamento) / orçamento
# 1e-9 (0.000000001): tolerância de ponto flutuante para ignorar resíduos de arredondamento
expected_roi = (F.col("revenue") - F.col("budget")) / F.col("budget")

cond_roi = has_budget & (F.abs(F.col("roi") - expected_roi) > 1e-9)

# agregação e contagem das violações
movie_genre_metric_violations_df = movie_genre_performance_df.select(
    F.sum(cond_eligibility.cast("int")).alias("eligibility_violations"),
    F.sum(cond_ineligible.cast("int")).alias("ineligible_metrics_violations"),
    F.sum(cond_release_year.cast("int")).alias("release_year_violations"),
    F.sum(cond_profit.cast("int")).alias("profit_violations"),
    F.sum(cond_roi.cast("int")).alias("roi_violations"),
)

display(movie_genre_metric_violations_df)

#### 4.2.3 Evidência analítica

Com a integridade do relacionamento filme × gênero e suas métricas validadas, o produto pode ser utilizado para comparar características comerciais entre gêneros.

Como um filme pode pertencer a múltiplos gêneros, as métricas financeiras são intencionalmente repetidas em cada associação. Por isso, esta análise prioriza quantidade de filmes e métricas médias, evitando interpretar somas entre gêneros como valores exclusivos do catálogo.

In [0]:
genre_performance_df = (
    movie_genre_performance_df.groupBy("genre_id", "genre_name")
    .agg(
        F.countDistinct("movie_id").alias("movie_count"),
        F.avg("budget").alias("avg_budget"),
        F.avg("revenue").alias("avg_revenue"),
        F.avg("profit").alias("avg_profit"),
        F.avg("roi").alias("avg_roi"),
        F.avg("vote_average").alias("avg_vote_average"),
    )
    .orderBy(F.col("movie_count").desc())
)

display(genre_performance_df)

> **Inspeção do recorte analítico.** A agregação inicial permite observar, por gênero, sua representatividade no catálogo e o comportamento das principais métricas disponíveis no produto.
>
> Para a comparação comercial, porém, `avg_roi` não será utilizado. A inspeção revela valores extremos em alguns gêneros, provocados pela
> sensibilidade do ROI a filmes com orçamentos positivos muito baixos. Uma média simples desses valores produziria uma comparação executiva poucorepresentativa.
>
> `avg_vote_average` também não será combinado com as métricas financeiras nesta análise, pois representa avaliação do público e responde a uma dimensão analítica diferente.
>
> A análise comercial seguinte utilizará apenas filmes elegíveis (`commercial_metrics_eligible = true`), garantindo uma população homogêneapara comparar orçamento, receita e lucro entre gêneros.

In [0]:
genre_commercial_performance_df = (
    movie_genre_performance_df.filter(F.col("commercial_metrics_eligible"))
    .groupBy("genre_id", "genre_name")
    .agg(
        F.countDistinct("movie_id").alias("eligible_movie_count"),
        F.avg("budget").alias("avg_budget"),
        F.avg("revenue").alias("avg_revenue"),
        F.avg("profit").alias("avg_profit"),
    )
    .orderBy(F.col("avg_revenue").desc())
)

display(genre_commercial_performance_df)

> **Recorte comercial.** A comparação a seguir considera somente filmes elegíveis para métricas comerciais. Dessa forma, orçamento, receita e lucro médios são calculados sobre uma população consistente dentro de cada gênero.

> A quantidade de filmes elegíveis é preservada como contexto para a leitura das médias, evitando comparar os resultados sem considerar a representatividade de cada gênero.

In [0]:
top_genre_revenue_df = genre_commercial_performance_df.orderBy(
    F.col("avg_revenue").desc()
).limit(10)

print("Quais gêneros apresentam maior receita média entre filmes comercialmente elegíveis?")
display(top_genre_revenue_df)

Databricks visualization. Run in Databricks to view.

### 4.3 movie_credit_participation

Consolida as participações de elenco e equipe técnica dos filmes em um único produto analítico, preservando a natureza de cada crédito.

O grão corresponde a uma participação identificada por filme, tipo de participação e crédito. Essa estrutura permite analisar pessoas e funções envolvidas nas produções sem misturar a semântica específica de `cast` e `crew`.

#### 4.3.1 Integridade do produto

Verifica se cada crédito permanece único dentro do filme e de seu tipo de participação e se a estrutura materializada corresponde ao contrato Gold definido para `movie_credit_participation`.

In [0]:
# busca e retorna o primeiro contrato da lista correspondente a 'movie_credit_participation'
movie_credit_contract = next(
    contract
    for contract in GOLD_CONTRACTS
    if contract.name == "movie_credit_participation"
)

# cardinalidade observada no produto
row_count = movie_credit_participation_df.count()

# regra: cada crédito deve ser único por filme e tipo de participação
duplicate_key_count = (
    movie_credit_participation_df.groupBy(*movie_credit_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# estrutura materializada observada
actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_credit_participation_df.schema.fields
]

# estrutura definida pelo contrato Gold
expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_credit_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError("movie_credit_participation possui créditos duplicados.")

if not schema_matches:
    raise RuntimeError(
        "movie_credit_participation não corresponde ao schema definido no contrato Gold."
    )

#### 4.3.2 Regras de participação

Valida a semântica que diferencia créditos de elenco (`cast`) e equipe técnica (`crew`).

Créditos de elenco podem possuir personagem e ordem de aparição, mas não departamento ou função técnica. Créditos de equipe técnica seguem a relação inversa: podem possuir departamento e função, mas não personagem ou ordem de elenco

In [0]:
# definição das condições de validação (regras booleanas)
is_cast = F.col("participation_type") == "cast"
is_crew = F.col("participation_type") == "crew"

# regra: participation_type deve aceitar somente cast ou crew
cond_invalid_participation_type = ~F.col("participation_type").isin("cast", "crew")

# regra: créditos de cast não podem possuir atributos exclusivos de crew
cond_cast_with_crew_fields = is_cast & (
    F.col("department").isNotNull() | F.col("job").isNotNull()
)

# regra: créditos de crew não podem possuir atributos exclusivos de cast
cond_crew_with_cast_fields = is_crew & (
    F.col("character").isNotNull() | F.col("cast_order").isNotNull()
)

# agregação e contagem das violações
movie_credit_violations_df = movie_credit_participation_df.select(
    F.sum(cond_invalid_participation_type.cast("int")).alias(
        "invalid_participation_type_violations"
    ),
    F.sum(cond_cast_with_crew_fields.cast("int")).alias(
        "cast_with_crew_fields_violations"
    ),
    F.sum(cond_crew_with_cast_fields.cast("int")).alias(
        "crew_with_cast_fields_violations"
    ),
)

display(movie_credit_violations_df)

#### 4.3.3 Evidência analítica

Com a integridade e a separação semântica entre elenco e equipe técnica validadas, o produto pode ser utilizado para analisar a composição dos créditos dos filmes.

A inspeção inicial observa a distribuição das participações entre `cast` e `crew` antes de aprofundar a análise em pessoas, departamentos ou funções.

In [0]:
credit_participation_profile_df = (
    movie_credit_participation_df.groupBy("participation_type")
    .agg(
        F.count("*").alias("participation_count"),
        F.countDistinct("movie_id").alias("movie_count"),
        F.countDistinct("person_id").alias("person_count"),
    )
    .orderBy(F.col("participation_count").desc())
)

display(credit_participation_profile_df)

> **Inspeção do perfil de créditos.** A equipe técnica (`crew`) concentra mais participações registradas que o elenco (`cast`), enquanto o elenco apresenta uma quantidade ligeiramente maior de pessoas distintas.
>
> Ambos os tipos de participação estão presentes em praticamente todo o catálogo, indicando que o produto oferece cobertura ampla para análises tanto de elenco quanto de funções técnicas.

In [0]:
"""
O recorte a seguir identifica as pessoas com maior número de créditos de `cast` no catálogo, permitindo observar quem aparece com maior recorrência nas produções representadas pelo produto.

A métrica representa quantidade de participações registradas, e não número de filmes distintos.
"""
top_cast_participation_df = (
    movie_credit_participation_df.filter(F.col("participation_type") == "cast")
    .groupBy("person_id", "person_name")
    .agg(
        F.count("*").alias("participation_count"),
        F.countDistinct("movie_id").alias("movie_count"),
    )
    .orderBy(
        F.col("participation_count").desc(),
        F.col("person_name"),
    )
    .limit(10)
)

display(top_cast_participation_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** Samuel L. Jackson apresenta a maior recorrência de elenco no catálogo, com 67 participações, seguido por Robert De Niro (57) e Bruce Willis (51).
>
> Para todas as pessoas do Top 10, a quantidade de participações coincide com a quantidade de filmes distintos. Nesse recorte, portanto, a recorrência observada representa presença em diferentes produções, e não múltiplos créditos da mesma pessoa dentro de um mesmo filme.

In [0]:
"""
O recorte a seguir observa como os créditos de `crew` se distribuem entre os departamentos registrados no catálogo.

A quantidade de participações representa créditos técnicos, enquanto a quantidade de filmes distintos fornece contexto sobre a abrangência de cada
departamento nas produções
"""

crew_department_profile_df = (
    movie_credit_participation_df.filter(F.col("participation_type") == "crew")
    .groupBy("department")
    .agg(
        F.count("*").alias("participation_count"),
        F.countDistinct("movie_id").alias("movie_count"),
        F.countDistinct("person_id").alias("person_count"),
    )
    .orderBy(F.col("participation_count").desc())
)

display(crew_department_profile_df)

In [0]:
"""
A inspeção seguinte aprofunda a composição de `crew` pelas funções (`job`) registradas no produto.

A quantidade de participações mostra a recorrência de cada função técnica, enquanto filmes e pessoas distintas ajudam a contextualizar sua abrangência no catálogo.
"""

crew_job_profile_df = (
    movie_credit_participation_df.filter(F.col("participation_type") == "crew")
    .groupBy("job")
    .agg(
        F.count("*").alias("participation_count"),
        F.countDistinct("movie_id").alias("movie_count"),
        F.countDistinct("person_id").alias("person_count"),
    )
    .orderBy(F.col("participation_count").desc())
)

display(crew_job_profile_df)

In [0]:
top_crew_job_df = crew_job_profile_df.orderBy(
    F.col("participation_count").desc()
).limit(10)

display(top_crew_job_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** `Producer` concentra o maior número de participações técnicas no catálogo, com 10.206 créditos, seguido por `Executive Producer` (6.177) e `Director` (5.166).
>
> A recorrência de créditos, porém, não equivale à abrangência entre filmes. `Director`, por exemplo, aparece em 4.773 filmes, enquanto `Producer`, apesar de possuir quase o dobro de participações, está presente em 3.780. O produto permite, portanto, analisar tanto a intensidade dos créditos técnicos quanto sua distribuição entre as produções.

### 4.4 movie_company_performance

Relaciona os filmes às suas produtoras e às respectivas métricas de desempenho, permitindo analisar o comportamento comercial das produções
associadas a cada empresa.

O grão corresponde à associação entre filme e produtora. Como um filme pode estar associado a múltiplas produtoras, suas métricas podem aparecer em mais de uma associação sem representar duplicidade do produto.

#### 4.4.1 Integridade do produto

Verifica se cada associação entre filme e produtora permanece única e se a estrutura materializada corresponde ao contrato Gold definido para `movie_company_performance`.

In [0]:
# busca e retorna o primeiro contrato da lista correspondente a 'movie_company_performance'
movie_company_contract = next(
    contract
    for contract in GOLD_CONTRACTS
    if contract.name == "movie_company_performance"
)

# cardinalidade observada no produto
row_count = movie_company_performance_df.count()

# regra: cada associação filme × produtora deve ser única
duplicate_key_count = (
    movie_company_performance_df.groupBy(*movie_company_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# estrutura materializada observada
actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_company_performance_df.schema.fields
]

# estrutura definida pelo contrato Gold
expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_company_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError(
        "movie_company_performance possui associações filme-produtora duplicadas."
    )

if not schema_matches:
    raise RuntimeError(
        "movie_company_performance não corresponde ao schema definido no contrato Gold."
    )

#### 4.4.2 Regras e métricas comerciais

Valida se as métricas comerciais preservadas em cada associação filme-produtora permanecem coerentes com as regras definidas para a Gold.

A elegibilidade depende de orçamento positivo. Para filmes elegíveis, lucro e ROI devem corresponder às métricas derivadas de orçamento e receita; para os demais, essas métricas devem permanecer nulas.

In [0]:
# conceitos auxiliares
has_budget = F.col("budget") > 0

# regra: elegibilidade comercial deve refletir orçamento positivo
cond_eligibility = F.col("commercial_metrics_eligible") != has_budget

# regra: filmes não elegíveis não devem possuir lucro ou ROI
cond_ineligible = ~F.col("commercial_metrics_eligible") & (
    F.col("profit").isNotNull() | F.col("roi").isNotNull()
)

# regra: release_year deve corresponder ao ano de release_date
cond_release_year = F.col("release_date").isNotNull() & (
    F.col("release_year").isNull() | (F.col("release_year") != F.year("release_date"))
)

# regra: lucro deve corresponder a receita - orçamento
cond_profit = F.col("commercial_metrics_eligible") & (
    F.col("profit").isNull() | (F.col("profit") != (F.col("revenue") - F.col("budget")))
)

# regra: ROI deve corresponder a (receita - orçamento) / orçamento
expected_roi = (F.col("revenue") - F.col("budget")) / F.col("budget")

cond_roi = F.col("commercial_metrics_eligible") & (
    F.col("roi").isNull() | (F.abs(F.col("roi") - expected_roi) > 1e-9)
)

movie_company_violations_df = movie_company_performance_df.select(
    F.sum(cond_eligibility.cast("int")).alias("eligibility_violations"),
    F.sum(cond_ineligible.cast("int")).alias("ineligible_metrics_violations"),
    F.sum(cond_release_year.cast("int")).alias("release_year_violations"),
    F.sum(cond_profit.cast("int")).alias("profit_violations"),
    F.sum(cond_roi.cast("int")).alias("roi_violations"),
)

display(movie_company_violations_df)

#### 4.4.3 Evidência analítica

Com a integridade e as métricas comerciais validadas, o produto pode ser utilizado para analisar o desempenho dos filmes associados às produtoras.

A inspeção inicial observa a representatividade das empresas no catálogo e o comportamento das principais métricas comerciais antes de definir os recortes executivos da análise.

In [0]:
"""
O recorte inicial consolida as associações por produtora, preservando a quantidade de filmes como contexto para interpretar as métricas médias.

As métricas comerciais são calculadas somente sobre filmes elegíveis, mantendo orçamento, receita e lucro sobre uma população consistente.
"""

company_commercial_performance_df = (
    movie_company_performance_df.filter(F.col("commercial_metrics_eligible"))
    .groupBy("company_id", "company_name")
    .agg(
        F.countDistinct("movie_id").alias("eligible_movie_count"),
        F.avg("budget").alias("avg_budget"),
        F.avg("revenue").alias("avg_revenue"),
        F.avg("profit").alias("avg_profit"),
    )
)

display(
    company_commercial_performance_df.orderBy(
        F.col("eligible_movie_count").desc()
    ).limit(20)
)

In [0]:
"""
Antes de comparar o desempenho médio das produtoras, esta inspeção observa a distribuição da quantidade de filmes comercialmente elegíveis por empresa.

O objetivo é avaliar a representatividade das produtoras e fundamentar um eventual recorte analítico sem transformar um limite arbitrário em regra do produto Gold.
"""

company_representativeness_df = company_commercial_performance_df.select(
    "eligible_movie_count"
).summary("count", "min", "25%", "50%", "75%", "90%", "95%", "max")

display(company_representativeness_df)

In [0]:
"""
A inspeção compara limites candidatos de representatividade para verificar quantas produtoras permaneceriam disponíveis em uma análise de desempenho médio.

Os limites são critérios exploratórios da EDA e não regras do contrato Gold.
"""

company_threshold_profile_df = company_commercial_performance_df.agg(
    F.sum((F.col("eligible_movie_count") >= 5).cast("int")).alias(
        "companies_with_5_plus_movies"
    ),
    F.sum((F.col("eligible_movie_count") >= 10).cast("int")).alias(
        "companies_with_10_plus_movies"
    ),
    F.sum((F.col("eligible_movie_count") >= 20).cast("int")).alias(
        "companies_with_20_plus_movies"
    ),
    F.sum((F.col("eligible_movie_count") >= 50).cast("int")).alias(
        "companies_with_50_plus_movies"
    ),
)

display(company_threshold_profile_df)

In [0]:
"""
Para a comparação executiva de desempenho médio, são consideradas produtoras com pelo menos 20 filmes comercialmente elegíveis.

O limite é um critério analítico desta EDA, fundamentado na distribuição
observada das produtoras. Não constitui regra do produto Gold.
"""

company_relevance_movie_count = 20

top_company_revenue_df = (
    company_commercial_performance_df.filter(
        F.col("eligible_movie_count") >= company_relevance_movie_count
    )
    .orderBy(F.col("avg_revenue").desc())
    .limit(10)
)

top_company_revenue_df.show(10, truncate=False)

In [0]:

# display(top_company_revenue_df)

# top_company_revenue_df.explain("formatted")


# Nota de investigação.** Durante a exploração, algumas renderizações com `display()` apresentaram latência elevada. A inspeção do plano físico mostrou leitura direta da tabela Gold, execução integralmente suportada por Photon e estatísticas disponíveis. A execução textual do mesmo dataframe confirmou o resultado sem indicar problema na transformação Spark. Não foi realizada otimização, pois a evidência não justificou alteração do processamento 


In [0]:
display(top_company_revenue_df)

Databricks visualization. Run in Databricks to view.

In [0]:
company_budget_revenue_df = company_commercial_performance_df.filter(
    F.col("eligible_movie_count") >= company_relevance_movie_count
).select(
    "company_name",
    "eligible_movie_count",
    "avg_budget",
    "avg_revenue",
    "avg_profit",
)

display(company_budget_revenue_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** O gráfico evidencia uma relação positiva entre orçamento médio e receita média entre as produtoras analisadas: empresas associadas a produções de maior orçamento tendem também a apresentar receitas médias mais elevadas.

> A dispersão dos pontos mostra, porém, que níveis semelhantes de investimento não resultam necessariamente no mesmo desempenho comercial. Algumas produtoras
alcançam receitas médias superiores às de outras com orçamento médio comparável, evidenciando diferenças no resultado obtido para patamares próximos de investimento.

> O recorte considera apenas produtoras com pelo menos 20 filmes comercialmente elegíveis, reduzindo a influência da longa cauda de empresas representadas por poucas produções. Esse limite é um critério analítico desta EDA e não uma regra do produto Gold.

### 4.5 movie_country_performance

Relaciona os filmes aos países de produção e às respectivas métricas de desempenho, permitindo analisar como a produção cinematográfica e seus resultados comerciais se distribuem entre países

O grão corresponde à associação entre filme e país de produção. Como um filme pode estar associado a múltiplos países, suas métricas podem aparecer em mais de uma associação sem representar duplicidade do produto

#### 4.5.1 Integridade do produto

Verifica se cada associação entre filme e pais permanece única e se a estrutura materializada corresponde ao contrato Gold definido para `movie_country_performance`.

In [0]:
# busca e retorna o primeiro contrato da lista correspondente a 'movie_country_performance'
country_contract = next(
    contract
    for contract in GOLD_CONTRACTS
    if contract.name == "movie_country_performance"
)

# cardinalidade observada no produto
row_count = movie_country_performance_df.count()

# regra: cada associação filme × país deve ser única
duplicate_key_count = (
    movie_country_performance_df.groupBy(*country_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# estrutura materializada observada
actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_country_performance_df.schema.fields
]

# estrutura definida pelo contrato Gold
expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in country_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError(
        "movie_country_performance possui associações filme-país duplicadas"
    )

if not schema_matches:
    raise RuntimeError(
        "movie_country_performance não corresponde ao schema definido no contrato Gold"
    )

#### 4.5.2 Regras e métricas comerciais

Verifica se as métricas comerciais associadas aos países preservam as regras definidas para os produtos Gold, elegibilidade comercial determinada pelo orçamento, derivação do ano de lançamento e consistência de lucro e ROI.

Como as métricas pertencem ao filme e são propagadas para cada associação filme–país, as mesmas regras devem permanecer válidas em todas as relações materializadas.

In [0]:
# regra auxiliar: métricas comerciais são elegíveis quando o orçamento é positivo
has_budget = F.col("budget") > 0

# violações das regras comerciais
eligibility_violation = F.col("commercial_metrics_eligible") != has_budget

ineligible_metrics_violation = (~F.col("commercial_metrics_eligible")) & (
    F.col("profit").isNotNull() | F.col("roi").isNotNull()
)

release_year_violation = F.col("release_date").isNotNull() & (
    F.col("release_year") != F.year("release_date")
)

profit_violation = F.col("commercial_metrics_eligible") & (
    F.col("profit") != (F.col("revenue") - F.col("budget"))
)

expected_roi = (F.col("revenue") - F.col("budget")) / F.col("budget")

roi_violation = F.col("commercial_metrics_eligible") & (
    F.abs(F.col("roi") - expected_roi) > 1e-9
)

country_commercial_validation_df = movie_country_performance_df.agg(
    F.sum(eligibility_violation.cast("int")).alias("eligibility_violations"),
    F.sum(ineligible_metrics_violation.cast("int")).alias(
        "ineligible_metrics_violations"
    ),
    F.sum(release_year_violation.cast("int")).alias("release_year_violations"),
    F.sum(profit_violation.cast("int")).alias("profit_violations"),
    F.sum(roi_violation.cast("int")).alias("roi_violations"),
)

display(country_commercial_validation_df)

#### 4.5.3 Evidência analítica

Com a integridade e as métricas comerciais validadas, o produto pode ser utilizado para analisar a distribuição das produções entre países e comparar o desempenho comercial dos filmes associados a cada mercado produtor.

A exploração inicial observa a representatividade dos países no catálogo e suas métricas comerciais antes de definir a visualização mais adequada para a evidência.

In [0]:
"""
O primeiro recorte mede quantos filmes distintos estão associados a cada país.

Como um filme pode possuir múltiplos países de produção, os valores representam participações por país e não devem ser somados para obter o total de filmes do catálogo.
"""

country_representation_df = (
    movie_country_performance_df.groupBy("country_code", "country_name")
    .agg(F.countDistinct("movie_id").alias("movie_count"))
    .orderBy(F.col("movie_count").desc())
)

display(country_representation_df)

country_representativeness_df = country_representation_df.select("movie_count").summary(
    "count", "min", "25%", "50%", "75%", "90%", "95%", "max"
)

display(country_representativeness_df)

country_threshold_profile_df = country_representation_df.agg(
    F.sum((F.col("movie_count") >= 5).cast("int")).alias(
        "countries_with_5_plus_movies"
    ),
    F.sum((F.col("movie_count") >= 10).cast("int")).alias(
        "countries_with_10_plus_movies"
    ),
    F.sum((F.col("movie_count") >= 20).cast("int")).alias(
        "countries_with_20_plus_movies"
    ),
    F.sum((F.col("movie_count") >= 50).cast("int")).alias(
        "countries_with_50_plus_movies"
    ),
)

display(country_threshold_profile_df)

In [0]:
"""
Para a comparação comercial entre países, são considerados países com pelo menos 20 filmes comercialmente elegíveis.

O limite é um critério analítico desta EDA, fundamentado na distribuição observada dos países. Não constitui regra do produto Gold.
"""

country_relevance_movie_count = 20

country_commercial_performance_df = (
    movie_country_performance_df.filter(F.col("commercial_metrics_eligible"))
    .groupBy("country_code", "country_name")
    .agg(
        F.countDistinct("movie_id").alias("eligible_movie_count"),
        F.avg("budget").alias("avg_budget"),
        F.avg("revenue").alias("avg_revenue"),
        F.avg("profit").alias("avg_profit"),
    )
    .filter(F.col("eligible_movie_count") >= country_relevance_movie_count)
    .orderBy(F.col("avg_revenue").desc())
)

display(country_commercial_performance_df)

Databricks visualization. Run in Databricks to view.

> **Interpretação.** Entre os países com pelo menos 20 filmes comercialmente elegíveis, a receita média apresenta diferenças relevantes entre os mercados de produção representados no catálogo.

> A Nova Zelândia ocupa o extremo superior da distribuição, enquanto países como Reino Unido, Estados Unidos, China e Japão também aparecem nas faixas mais elevadas de receita média. A visualização evidencia que maior presença no catálogo não implica necessariamente maior receita média: os Estados Unidos possuem uma população de filmes muito superior à dos demais países, mas não lideram essa métrica.

> O recorte considera apenas países com pelo menos 20 filmes comercialmente elegíveis, reduzindo a influência de médias calculadas sobre poucas produções.
Como um mesmo filme pode estar associado a mais de um país, os resultados representam associações de produção e não mercados mutuamente exclusivos. Esse limite é um critério analítico desta EDA e não uma regra do produto Gold.

### 4.6 movie_language_profile

Representa os idiomas associados aos filmes distinguindo o papel desempenhado por cada idioma: idioma original da obra ou idioma falado na produção.

O grão corresponde à combinação entre filme, papel do idioma e código do idioma. Assim, um mesmo filme pode possuir múltiplas associações linguísticas sem representar duplicidade do produto.

#### 4.6.1 Integridade do produto

Verifica se cada associação entre filme, papel do idioma e idioma permanece única e se a estrutura materializada corresponde ao contrato Gold definido para `movie_language_profile`.

In [0]:
# busca e retorna o primeiro contrato da lista correspondente a 'movie_language_profile'
language_contract = next(
    contract for contract in GOLD_CONTRACTS if contract.name == "movie_language_profile"
)

# cardinalidade observada no produto
row_count = movie_language_profile_df.count()

# regra: cada associação filme × papel do idioma × idioma deve ser única
duplicate_key_count = (
    movie_language_profile_df.groupBy(*language_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# estrutura materializada observada
actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_language_profile_df.schema.fields
]

# estrutura definida pelo contrato Gold
expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in language_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError(
        "movie_language_profile possui associações filme-papel-idioma duplicadas"
    )

if not schema_matches:
    raise RuntimeError(
        "movie_language_profile não corresponde ao schema definido no contrato Gold"
    )

#### 4.6.2 Regras semânticas e métricas comerciais

Verifica se os papéis linguísticos permanecem restritos às categorias previstas pelo produto e se as métricas comerciais associadas aos filmes preservam as regras definidas para a camada Gold.

As validações combinam, portanto, a semântica específica do perfil linguístico com as regras comerciais compartilhadas pelos produtos de desempenho.

In [0]:
# regra semântica: somente os papéis linguísticos previstos são válidos
language_role_violation = ~F.col("language_role").isin("original", "spoken")

# regra auxiliar: métricas comerciais são elegíveis quando o orçamento é positivo
has_budget = F.col("budget") > 0

# violações das regras comerciais
eligibility_violation = F.col("commercial_metrics_eligible") != has_budget

ineligible_metrics_violation = (~F.col("commercial_metrics_eligible")) & (
    F.col("profit").isNotNull() | F.col("roi").isNotNull()
)

release_year_violation = F.col("release_date").isNotNull() & (
    F.col("release_year") != F.year("release_date")
)

profit_violation = F.col("commercial_metrics_eligible") & (
    F.col("profit") != (F.col("revenue") - F.col("budget"))
)

expected_roi = (F.col("revenue") - F.col("budget")) / F.col("budget")

roi_violation = F.col("commercial_metrics_eligible") & (
    F.abs(F.col("roi") - expected_roi) > 1e-9
)

language_profile_validation_df = movie_language_profile_df.agg(
    F.sum(language_role_violation.cast("int")).alias("language_role_violations"),
    F.sum(eligibility_violation.cast("int")).alias("eligibility_violations"),
    F.sum(ineligible_metrics_violation.cast("int")).alias(
        "ineligible_metrics_violations"
    ),
    F.sum(release_year_violation.cast("int")).alias("release_year_violations"),
    F.sum(profit_violation.cast("int")).alias("profit_violations"),
    F.sum(roi_violation.cast("int")).alias("roi_violations"),
)

display(language_profile_validation_df)

#### 4.6.3 Evidência analítica

Com a integridade e as regras semânticas validadas, o produto pode ser utilizado para analisar o perfil linguístico dos filmes distinguindo idiomas originais dos idiomas falados nas produções.

A exploração inicial observa a representatividade dos idiomas em cada papel antes de comparar seus perfis e definir a visualização mais adequada.

In [0]:
language_representation_df = (
    movie_language_profile_df.groupBy(
        "language_role",
        "language_code",
        "language_name",
    )
    .agg(F.countDistinct("movie_id").alias("movie_count"))
    .orderBy(
        "language_role",
        F.col("movie_count").desc(),
    )
)

print("Quál é a cobertura, a diversidade de idiomas e o número de associações em cada papel linguistico?")
display(language_representation_df)

language_role_profile_df = (
    movie_language_profile_df.groupBy("language_role")
    .agg(
        F.countDistinct("movie_id").alias("movie_count"),
        F.countDistinct("language_code").alias("language_count"),
        F.count("*").alias("association_count"),
    )
    .orderBy("language_role")
)

print("Como os idiomas se distribuem entreo os papéis original e spoke?")
display(language_role_profile_df)

original_language_top_df = (
    language_representation_df.filter(F.col("language_role") == "original")
    .orderBy(F.col("movie_count").desc())
    .limit(5)
)

print("Quais são os 5 idiomas originais mais representativos no catalogo?")
display(original_language_top_df)

In [0]:
# cinco idiomas originais mais representados
top_original_language_codes = [
    row["language_code"] for row in original_language_top_df.collect()
]

original_language_composition_df = (
    movie_language_profile_df.filter(F.col("language_role") == "original")
    .withColumn(
        "language_group",
        F.when(
            F.col("language_code").isin(top_original_language_codes),
            F.col("language_code"),
        ).otherwise(F.lit("Outros")),
    )
    .groupBy("language_group")
    .agg(F.countDistinct("movie_id").alias("movie_count"))
    .orderBy(F.col("movie_count").desc())
)

print("como os filmes do catálogo se distribuem entre os principais idiomas originais?")

display(original_language_composition_df)

Databricks visualization. Run in Databricks to view.

**Interpretação.** O catálogo apresenta forte concentração no inglês como idioma original: 4.505 dos 4.803 filmes possuem `en` como código de idioma, correspondendo a aproximadamente 93,8% do total.

Os demais idiomas possuem participação substancialmente menor. O francês é o segundo idioma original mais frequente, com 70 filmes, seguido por espanhol, alemão e chinês, enquanto os outros idiomas reunidos somam 142 filmes.

A diferença de escala evidencia que, embora o produto represente 37 idiomas originais distintos, essa diversidade ocorre dentro de um catálogo fortemente
concentrado em produções originalmente em inglês. Por isso, análises comparativas entre idiomas devem considerar a grande diferença de representatividade entre essas populações.

In [0]:
non_english_original_languages_df = (
    language_representation_df.filter(
        (F.col("language_role") == "original") & (F.col("language_code") != "en")
    )
    .orderBy(F.col("movie_count").desc())
    .limit(10)
    .select(
        "language_code",
        "movie_count",
    )
)

print(
    "Pergunta analítica: quais idiomas originais, além do inglês, "
    "são mais representados no catálogo?"
)

display(non_english_original_languages_df)

Databricks visualization. Run in Databricks to view.

**Interpretação.** Ao excluir o inglês da comparação, torna-se visível a distribuição dos demais idiomas originais presentes no catálogo. O francês é o mais representado nesse recorte, com 70 filmes, seguido pelo espanhol, com 32, e por chinês e alemão, ambos com 27 filmes.

A partir dessas primeiras posições, a frequência diminui progressivamente: hindi, japonês, italiano, cantonês, russo e coreano aparecem com populações ainda menores. Não se observa, portanto, um segundo idioma com presença próxima à do inglês; a parcela não inglesa está distribuída entre diversos idiomas com representatividade individual relativamente baixa.

Esta visualização complementa a anterior ao retirar da escala a forte concentração do inglês e tornar comparável a diversidade existente entre os idiomas originais menos frequentes.

### 4.7 movie_keyword_performance

Relaciona os filmes às palavras-chave que descrevem temas, elementos narrativos e características associadas às obras, preservando também suas métricas de
desempenho.

O grão corresponde à associação entre filme e palavra-chave. Como um filme pode possuir múltiplas palavras-chave, suas métricas podem aparecer em várias associações sem representar duplicidade do produto.

#### 4.7.1 Integridade do produto

Verifica se cada associação entre filme e palavra-chave permanece única e se a estrutura materializada corresponde ao contrato Gold definido para `movie_keyword_performance`.

In [0]:
# busca e retorna o primeiro contrato da lista correspondente a 'movie_keyword_performance'
keyword_contract = next(
    contract
    for contract in GOLD_CONTRACTS
    if contract.name == "movie_keyword_performance"
)

# cardinalidade observada no produto
row_count = movie_keyword_performance_df.count()

# regra: cada associação filme × palavra-chave deve ser única
duplicate_key_count = (
    movie_keyword_performance_df.groupBy(*keyword_contract.key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# estrutura materializada observada
actual_schema = [
    (field.name, field.dataType.simpleString())
    for field in movie_keyword_performance_df.schema.fields
]

# estrutura definida pelo contrato Gold
expected_schema = [
    (field.name, field.dataType.simpleString())
    for field in keyword_contract.schema.fields
]

schema_matches = actual_schema == expected_schema

print(f"Registros: {row_count}")
print(f"Chaves duplicadas: {duplicate_key_count}")
print(f"Schema conforme contrato: {schema_matches}")

if duplicate_key_count > 0:
    raise RuntimeError(
        "movie_keyword_performance possui associações filme-palavra-chave duplicadas"
    )

if not schema_matches:
    raise RuntimeError(
        "movie_keyword_performance não corresponde ao schema definido no contrato Gold"
    )

#### 4.7.2 Regras e métricas comerciais

Verifica se as métricas comerciais associadas às palavras-chave preservam as regras definidas para os produtos Gold: elegibilidade determinada pelo orçamento, derivação do ano de lançamento e consistência de lucro e ROI

Como as métricas pertencem ao filme e são propagadas para cada associação filme–palavra-chave, as mesmas regras devem permanecer válidas em todas as relações materializadas

In [0]:
# regra auxiliar: métricas comerciais são elegíveis quando o orçamento é positivo
has_budget = F.col("budget") > 0

# violações das regras comerciais
eligibility_violation = F.col("commercial_metrics_eligible") != has_budget

ineligible_metrics_violation = (~F.col("commercial_metrics_eligible")) & (
    F.col("profit").isNotNull() | F.col("roi").isNotNull()
)

release_year_violation = F.col("release_date").isNotNull() & (
    F.col("release_year") != F.year("release_date")
)

profit_violation = F.col("commercial_metrics_eligible") & (
    F.col("profit") != (F.col("revenue") - F.col("budget"))
)

expected_roi = (F.col("revenue") - F.col("budget")) / F.col("budget")

roi_violation = F.col("commercial_metrics_eligible") & (
    F.abs(F.col("roi") - expected_roi) > 1e-9
)

keyword_commercial_validation_df = movie_keyword_performance_df.agg(
    F.sum(eligibility_violation.cast("int")).alias("eligibility_violations"),
    F.sum(ineligible_metrics_violation.cast("int")).alias(
        "ineligible_metrics_violations"
    ),
    F.sum(release_year_violation.cast("int")).alias("release_year_violations"),
    F.sum(profit_violation.cast("int")).alias("profit_violations"),
    F.sum(roi_violation.cast("int")).alias("roi_violations"),
)

print(
    "As associações filme–palavra-chave preservam corretamente as regras comerciais do produto Gold?"
)

display(keyword_commercial_validation_df)

#### 4.7.3 Evidência analítica

Com a integridade e as métricas comerciais validadas, o produto pode ser utilizado para analisar a presença de temas e características associados aos filmes e relacioná-los ao desempenho das produções

A exploração inicial observa a frequência das palavras-chave no catálogo antes de comparar métricas de desempenho, evitando conclusões baseadas em termos representados por poucas produções

In [0]:
keyword_representation_df = (
    movie_keyword_performance_df.groupBy("keyword_id", "keyword_name")
    .agg(F.countDistinct("movie_id").alias("movie_count"))
    .orderBy(F.col("movie_count").desc())
)

print("Quais palavras-chave são mais recorrentes entre os filmes do catálogo?")

display(keyword_representation_df)

In [0]:
keyword_distribution_df = keyword_representation_df.select("movie_count").summary(
    "count", "min", "25%", "50%", "75%", "90%", "95%", "max"
)

print("Como se distribui a quantidade de filmes associados a cada palavra-chave?")

display(keyword_distribution_df)

In [0]:
keyword_threshold_profile_df = keyword_representation_df.agg(
    F.sum((F.col("movie_count") >= 5).cast("int")).alias("keywords_with_5_plus_movies"),
    F.sum((F.col("movie_count") >= 10).cast("int")).alias(
        "keywords_with_10_plus_movies"
    ),
    F.sum((F.col("movie_count") >= 20).cast("int")).alias(
        "keywords_with_20_plus_movies"
    ),
    F.sum((F.col("movie_count") >= 50).cast("int")).alias(
        "keywords_with_50_plus_movies"
    ),
)

print(
    "Quantas palavras-chave permanecem representadas quando exigimos diferentes quantidades mínimas de filmes?"
)

display(keyword_threshold_profile_df)

In [0]:
keyword_relevance_movie_count = 50

keyword_commercial_performance_df = (
    movie_keyword_performance_df.filter(F.col("commercial_metrics_eligible"))
    .groupBy("keyword_id", "keyword_name")
    .agg(
        F.countDistinct("movie_id").alias("eligible_movie_count"),
        F.avg("budget").alias("avg_budget"),
        F.avg("revenue").alias("avg_revenue"),
        F.avg("profit").alias("avg_profit"),
    )
    .filter(F.col("eligible_movie_count") >= keyword_relevance_movie_count)
    .orderBy(F.col("avg_revenue").desc())
)

print(
    "Entre palavras-chave associadas a pelo menos 50 filmes comercialmente elegíveis, quais apresentam maior receita média?"
)

display(keyword_commercial_performance_df)

Databricks visualization. Run in Databricks to view.

**Interpretação.** O gráfico de dispersão evidencia uma relação positiva entre orçamento médio e receita média entre palavras-chave com representatividade analítica suficiente no catálogo. No extremo superior direito destacam-se termos como `based on comic book`, `3d` e `superhero`, associados a orçamentos médios superiores a US$ 110 milhões e receitas médias próximas ou superiores a US$ 400 milhões.

No extremo oposto, `independent film` aparece associado a uma escala comercial substancialmente menor, com orçamento médio inferior a US$ 8 milhões e receitamédia também próxima desse patamar. Entre esses extremos, a dispersão mostra diferentes níveis de receita para faixas semelhantes de orçamento, indicando que maior investimento médio não determina, isoladamente, o desempenho comercial observado.

O recorte considera apenas palavras-chave associadas a pelo menos 50 filmes comercialmente elegíveis. Esse limite reduz a influência da longa cauda de termos representados por poucas produções e constitui um critério analítico desta EDA, não uma regra do produto Gold